In [ ]:
import os

In [ ]:
pip install laya

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 16.9 MB/s eta 0:00:00


In [ ]:
from laya import Router

router = Router()  # downloads a checkpoint on first use; Router(preload=True) loads all three up front

state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
questions = {
    "department": {"type": "choice", "instructions": "Which department should handle this?",
                   "criteria": {"billing": "invoices, payments, refunds",
                                "technical": "bugs, outages, system errors",
                                "other": "everything else"}},
    "urgency": {"type": "score", "instructions": "How urgent is this?",
                "criteria": ["not urgent", "soon", "blocking"]},
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"},
}

result = router.predict(state, questions)
print(result["answers"]["department"]["choice"])  # billing
print(result["answers"]["churn_risk"]["noul"])    # probability the answer is yes
print(result["routing"]["model"])                 # english


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


billing
0.879
english


In [ ]:
hcp_state = {
    "hcp_profile": {
        "specialty": "Oncology",
        "decile": 9,
        "territory": "Northeast-A",
        "key_opinion_leader": True
    },
    "rx_dynamics": {
        "trx_volume_3m": 142,
        "brand_growth_pct": -12.5,
        "competitor_growth_pct": 18.0,
        "formulary_tier": "Preferred Tier 2"
    },
    "recent_interactions": [
        "Rep visit (F2F) 45 days ago: HCP expressed concerns over adverse cardiac events and patient adherence.",
        "Opened last 2 digital newsletters covering ASCO trial data.",
        "Unsubscribed from generic promotional product emails."
    ]
}

In [ ]:
hcp_targeting_questions = {
    # 1. HCP Prioritization
    "rep_visit_priority": {
        "type": "choice",
        "instructions": "What priority tier should be assigned for field sales representative scheduling?",
        "criteria": {
            "critical_immediate": "High decile with dropping brand share or competitive threat; requires urgent rep intervention within 7 days.",
            "maintain_regular": "Stable prescriber meeting targets; scheduled for standard cycle cadences.",
            "digital_only": "Low decile, unresponsive to reps, or non-access HCP best served by digital touchpoints.",
            "deprioritized": "Strict no-see policy or low adoption potential."
        }
    },
    "urgent_churn_risk": {
        "type": "noul",
        "instructions": "Is this HCP actively shifting switch-share to competitor therapies?"
    },

    # 2. Channel Affinity
    "optimal_channel": {
        "type": "choice",
        "instructions": "Which channel should be selected for the next touchpoint?",
        "criteria": {
            "in_person_rep": "High-touch clinical discussion, objection handling, or detailing required.",
            "rep_triggered_email": "Follow-up requested or high digital open-rates paired with active rep relationship.",
            "msl_medical_inquiry": "KOL seeking deep scientific, off-label, or trial publication data.",
            "self_serve_portal": "Prefers on-demand portal resources or webinar invitations."
        }
    },

    # 3. Content Affinity
    "content_focus": {
        "type": "choice",
        "instructions": "Which content theme addresses the HCP's primary barrier or interest?",
        "criteria": {
            "clinical_efficacy": "Comparative endpoints, progression-free survival, head-to-head trials.",
            "safety_tolerability": "Adverse event management, drug interactions, contraindications.",
            "patient_support_access": "Copay assistance, prior authorization guidelines, adherence programs.",
            "dosing_administration": "Infusion schedules, dose escalation, monitoring protocols."
        }
    },

    # 4. Message Affinity / Framing
    "framing_angle": {
        "type": "choice",
        "instructions": "What message tone will resonate best with this HCP?",
        "criteria": {
            "evidence_first": "Data-heavy, statistical citations, guideline committee consensus.",
            "patient_centric": "Quality-of-life benefits, real-world registry evidence, patient case studies.",
            "practice_efficiency": "Ease of stocking, simplified clinic administration, nursing support."
        }
    }
}

In [ ]:
import laya

# Initialize the model (using English ModernBERT-large backbone)
# For multilingual markets, pass subfolder="multilingual"
agent = laya.load("convaiinnovations/laya")

# Single forward pass for all 4 dimensions (~35 ms)
result = agent.predict(hcp_state, hcp_targeting_questions)
answers = result["answers"]

print("--- Decision Engine Output ---")
print(f"Priority Tier   : {answers['rep_visit_priority']['choice']}")
print(f"Churn Risk      : {answers['urgent_churn_risk']['noul']:.2%}")
print(f"Optimal Channel : {answers['optimal_channel']['choice']}")
print(f"Content Focus   : {answers['content_focus']['choice']}")
print(f"Message Framing : {answers['framing_angle']['choice']}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:2143: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


--- Decision Engine Output ---
Priority Tier   : maintain_regular
Churn Risk      : 4.88%
Optimal Channel : rep_triggered_email
Content Focus   : clinical_efficacy
Message Framing : evidence_first


In [ ]:
result = router.predict(hcp_state, hcp_targeting_questions)
answers = result["answers"]

print("--- Decision Engine Output ---")
print(f"Priority Tier   : {answers['rep_visit_priority']['choice']}")
print(f"Churn Risk      : {answers['urgent_churn_risk']['noul']:.2%}")
print(f"Optimal Channel : {answers['optimal_channel']['choice']}")
print(f"Content Focus   : {answers['content_focus']['choice']}")
print(f"Message Framing : {answers['framing_angle']['choice']}")

--- Decision Engine Output ---
Priority Tier   : maintain_regular
Churn Risk      : 4.88%
Optimal Channel : rep_triggered_email
Content Focus   : clinical_efficacy
Message Framing : evidence_first


# HCP profile creation

In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.5 MB/s eta 0:00:00


In [ ]:
import json
import random
import numpy as np
from faker import Faker

fake = Faker("en_US")
np.random.seed(42)
random.seed(42)

SPECIALTIES = [
    "Medical Oncology", "Hematology Oncology", "Cardiology",
    "Endocrinology", "Rheumatology", "Neurology", "Immunology"
]

TERRITORIES = [
    "NE-Boston", "NE-NY-Metro", "SE-Atlanta", "MW-Chicago",
    "SW-Dallas", "WC-SF-Bay", "WC-LA-Metro", "NW-Seattle"
]

FORMULARY_TIERS = [
    "Preferred Tier 1", "Preferred Tier 2",
    "Non-Preferred Tier 3", "Prior Authorization Required"
]

INTERACTION_TEMPLATES = [
    "Rep visited {days} days ago: HCP noted concerns over patient tolerability and gastrointestinal adverse events.",
    "Attended regional symposium {days} days ago; expressed keen interest in 3-year progression-free survival data.",
    "Rep detailing note: Refused in-person meeting citing strict hospital no-see protocol; requested digital summary.",
    "Email interaction: Opened clinical trial newsletter; clicked link on head-to-head comparative efficacy study.",
    "MSL inquiry logged {days} days ago: Asked technical questions on biomarker sub-group analyses and dosing schedules.",
    "Recent field call: Mentions patient pushback on high copay costs; inquiring about patient co-pay cards and foundation support.",
    "Interaction note: High prescriber of competitor therapy; skeptical of switching stable patients without superior real-world evidence.",
    "Webinar attendee: Logged into live CME session on next-generation therapeutic combinations."
]

def generate_hcp_cohort(n_hcps=1000):
    cohort = []

    for i in range(1, n_hcps + 1):
        hcp_id = f"HCP_{i:04d}"
        specialty = random.choice(SPECIALTIES)
        decile = int(np.random.choice(range(1, 11), p=[0.05, 0.05, 0.08, 0.10, 0.12, 0.15, 0.15, 0.12, 0.10, 0.08]))
        is_kol = bool(decile >= 8 and random.random() < 0.25)

        # Correlate baseline volume with decile
        base_trx = int(np.clip(np.random.normal(loc=decile * 15, scale=8), 5, 200))
        brand_growth = round(float(np.random.normal(loc=-1.5 if decile > 6 else 2.0, scale=12.0)), 1)
        competitor_growth = round(float(np.random.normal(loc=4.0 if brand_growth < 0 else -1.0, scale=8.0)), 1)

        # CRM engagement and compliance flags
        days_since_last_call = int(np.random.exponential(scale=28)) + 1
        email_opt_in = random.random() < 0.72
        access_level = random.choices(["Open Access", "Restricted Access", "No-See"], weights=[0.55, 0.35, 0.10])[0]

        # Generate 1 to 3 recent CRM note entries
        num_notes = random.randint(1, 3)
        recent_notes = [
            random.choice(INTERACTION_TEMPLATES).format(days=random.randint(5, 90))
            for _ in range(num_notes)
        ]

        # Structure payload matching Laya's state input format
        hcp_record = {
            "hcp_id": hcp_id,
            "hcp_name": f"Dr. {fake.first_name()} {fake.last_name()}, MD",
            "metadata": {
                "specialty": specialty,
                "decile": decile,
                "territory": random.choice(TERRITORIES),
                "key_opinion_leader": is_kol,
                "access_level": access_level,
                "email_opt_in": email_opt_in,
                "days_since_last_call": days_since_last_call
            },
            # This 'laya_state' block is what you pass directly to router.predict(state, questions)
            "laya_state": {
                "profile": {
                    "specialty": specialty,
                    "decile": decile,
                    "access_level": access_level,
                    "is_kol": is_kol
                },
                "rx_dynamics": {
                    "trx_volume_3m": base_trx,
                    "brand_growth_pct": brand_growth,
                    "competitor_growth_pct": competitor_growth,
                    "formulary_tier": random.choice(FORMULARY_TIERS)
                },
                "recent_crm_notes": recent_notes
            }
        }
        cohort.append(hcp_record)

    return cohort

if __name__ == "__main__":
    hcps = generate_hcp_cohort(1000)

    # Save to JSON
    output_path = "hcp_cohort_1000.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(hcps, f, indent=2)

    print(f"Generated {len(hcps)} HCPs successfully and saved to '{output_path}'.")

Generated 1000 HCPs successfully and saved to 'hcp_cohort_1000.json'.


In [ ]:
import json
import laya
from tqdm import tqdm

# 1. Load generated dataset
with open("hcp_cohort_1000.json", "r") as f:
    cohort = json.load(f)

# 2. Define the decision questions
questions = {
    "rep_visit_priority": {
        "type": "choice",
        "instructions": "What priority tier should be assigned for sales scheduling?",
        "criteria": {
            "critical_immediate": "High decile with dropping brand share or competitor churn; contact in 7 days.",
            "maintain_regular": "Stable prescriber meeting targets; scheduled for standard cycle cadence.",
            "digital_only": "Low decile, unresponsive to reps, or non-access HCP best served digitally.",
            "deprioritized": "Strict no-see policy or low adoption potential."
        }
    },
    "urgent_churn_risk": {
        "type": "noul",
        "instructions": "Is this HCP actively shifting switch-share to competitor therapies?"
    },
    "optimal_channel": {
        "type": "choice",
        "instructions": "Which channel should be selected for the next touchpoint?",
        "criteria": {
            "in_person_rep": "High-touch clinical discussion or objection handling required.",
            "rep_triggered_email": "Follow-up requested or high digital open-rates.",
            "msl_medical_inquiry": "KOL seeking deep scientific, off-label, or trial publication data.",
            "self_serve_portal": "Prefers on-demand portal resources or webinar invitations."
        }
    },
    "content_focus": {
        "type": "choice",
        "instructions": "Which content theme addresses the primary barrier or interest?",
        "criteria": {
            "clinical_efficacy": "Comparative endpoints, survival curves, head-to-head trials.",
            "safety_tolerability": "Adverse event management, contraindications, toxicity mitigation.",
            "patient_support_access": "Copay assistance, prior authorizations, patient assistance programs.",
            "dosing_administration": "Infusion schedules, dose modifications, clinic handling protocols."
        }
    }
}

# 3. Load Laya model
# agent = laya.load("convaiinnovations/laya")

# 4. Process all HCPs
results = []
for hcp in tqdm(cohort, desc="Scoring HCPs"):
    state = hcp["laya_state"]

    decision = router.predict_batch(state, questions)

    results.append({
        "hcp_id": hcp["hcp_id"],
        "name": hcp["hcp_name"],
        "decisions": {
            "priority": decision["answers"]["rep_visit_priority"]["choice"],
            "churn_probability": round(decision["answers"]["urgent_churn_risk"]["noul"], 4),
            "channel": decision["answers"]["optimal_channel"]["choice"],
            "content": decision["answers"]["content_focus"]["choice"]
        }
    })

# 5. Save scored cohort
with open("hcp_scored_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Batch inference complete for 1,000 HCPs.")

Scoring HCPs:   0%|          | 0/1000 [00:00<?, ?it/s]


TypeError: requests must be a sequence of request dictionaries

In [ ]:
test = router.predict(cohort[0], questions)
test

{'model': 'laya-rl-agent',
 'answers': {'rep_visit_priority': {'type': 'choice',
   'choice': 'digital_only',
   'probabilities': {'critical_immediate': 0.1684,
    'maintain_regular': 0.302,
    'digital_only': 0.3528,
    'deprioritized': 0.1768},
   'confidence': 0.0366,
   'answer_confidence': 0.3528,
   'action': {'act_probability': 1.0}},
  'urgent_churn_risk': {'type': 'noul',
   'noul': 0.1358,
   'confidence': 0.8642,
   'answer_confidence': 0.8642,
   'action': {'act_probability': 1.0}},
  'optimal_channel': {'type': 'choice',
   'choice': 'rep_triggered_email',
   'probabilities': {'in_person_rep': 0.1615,
    'rep_triggered_email': 0.4965,
    'msl_medical_inquiry': 0.2181,
    'self_serve_portal': 0.124},
   'confidence': 0.1106,
   'answer_confidence': 0.4965,
   'action': {'act_probability': 1.0}},
  'content_focus': {'type': 'choice',
   'choice': 'clinical_efficacy',
   'probabilities': {'clinical_efficacy': 0.8516,
    'safety_tolerability': 0.0569,
    'patient_suppo

In [ ]:
test['answers']

{'rep_visit_priority': {'type': 'choice',
  'choice': 'digital_only',
  'probabilities': {'critical_immediate': 0.1684,
   'maintain_regular': 0.302,
   'digital_only': 0.3528,
   'deprioritized': 0.1768},
  'confidence': 0.0366,
  'answer_confidence': 0.3528,
  'action': {'act_probability': 1.0}},
 'urgent_churn_risk': {'type': 'noul',
  'noul': 0.1358,
  'confidence': 0.8642,
  'answer_confidence': 0.8642,
  'action': {'act_probability': 1.0}},
 'optimal_channel': {'type': 'choice',
  'choice': 'rep_triggered_email',
  'probabilities': {'in_person_rep': 0.1615,
   'rep_triggered_email': 0.4965,
   'msl_medical_inquiry': 0.2181,
   'self_serve_portal': 0.124},
  'confidence': 0.1106,
  'answer_confidence': 0.4965,
  'action': {'act_probability': 1.0}},
 'content_focus': {'type': 'choice',
  'choice': 'clinical_efficacy',
  'probabilities': {'clinical_efficacy': 0.8516,
   'safety_tolerability': 0.0569,
   'patient_support_access': 0.0538,
   'dosing_administration': 0.0377},
  'confid

In [ ]:
cohort[0]

{'hcp_id': 'HCP_0001',
 'hcp_name': 'Dr. Walter Garcia, MD',
 'metadata': {'specialty': 'Neurology',
  'decile': 5,
  'territory': 'NE-NY-Metro',
  'key_opinion_leader': False,
  'access_level': 'Restricted Access',
  'email_opt_in': True,
  'days_since_last_call': 35},
 'laya_state': {'profile': {'specialty': 'Neurology',
   'decile': 5,
   'access_level': 'Restricted Access',
   'is_kol': False},
  'rx_dynamics': {'trx_volume_3m': 66,
   'brand_growth_pct': 5.8,
   'competitor_growth_pct': 1.2,
   'formulary_tier': 'Preferred Tier 1'},
  'recent_crm_notes': ['Email interaction: Opened clinical trial newsletter; clicked link on head-to-head comparative efficacy study.']}}

In [ ]:
test_state = [cohort[0], cohort[1], cohort[2]]
test_state

[{'hcp_id': 'HCP_0001',
  'hcp_name': 'Dr. Walter Garcia, MD',
  'metadata': {'specialty': 'Neurology',
   'decile': 5,
   'territory': 'NE-NY-Metro',
   'key_opinion_leader': False,
   'access_level': 'Restricted Access',
   'email_opt_in': True,
   'days_since_last_call': 35},
  'laya_state': {'profile': {'specialty': 'Neurology',
    'decile': 5,
    'access_level': 'Restricted Access',
    'is_kol': False},
   'rx_dynamics': {'trx_volume_3m': 66,
    'brand_growth_pct': 5.8,
    'competitor_growth_pct': 1.2,
    'formulary_tier': 'Preferred Tier 1'},
   'recent_crm_notes': ['Email interaction: Opened clinical trial newsletter; clicked link on head-to-head comparative efficacy study.']}},
 {'hcp_id': 'HCP_0002',
  'hcp_name': 'Dr. William Alexander, MD',
  'metadata': {'specialty': 'Rheumatology',
   'decile': 1,
   'territory': 'NE-Boston',
   'key_opinion_leader': False,
   'access_level': 'Open Access',
   'email_opt_in': True,
   'days_since_last_call': 6},
  'laya_state': {'pro

In [ ]:

laya_batch_input = [
    {"state": hcp, "questions": questions}
    for hcp in test_state
]
test1 = router.predict_batch(laya_batch_input)

In [ ]:
len(test1)

3

In [ ]:
for i in test1:
  print(i['answers'])
  print()

{'rep_visit_priority': {'type': 'choice', 'choice': 'digital_only', 'probabilities': {'critical_immediate': 0.1684, 'maintain_regular': 0.302, 'digital_only': 0.3528, 'deprioritized': 0.1768}, 'confidence': 0.0366, 'answer_confidence': 0.3528, 'action': {'act_probability': 1.0}}, 'urgent_churn_risk': {'type': 'noul', 'noul': 0.1358, 'confidence': 0.8642, 'answer_confidence': 0.8642, 'action': {'act_probability': 1.0}}, 'optimal_channel': {'type': 'choice', 'choice': 'rep_triggered_email', 'probabilities': {'in_person_rep': 0.1615, 'rep_triggered_email': 0.4965, 'msl_medical_inquiry': 0.2181, 'self_serve_portal': 0.124}, 'confidence': 0.1106, 'answer_confidence': 0.4965, 'action': {'act_probability': 1.0}}, 'content_focus': {'type': 'choice', 'choice': 'clinical_efficacy', 'probabilities': {'clinical_efficacy': 0.8516, 'safety_tolerability': 0.0569, 'patient_support_access': 0.0538, 'dosing_administration': 0.0377}, 'confidence': 0.581, 'answer_confidence': 0.8516, 'action': {'act_proba

In [ ]:
def chunk_data(data, chunk_size=32):
    """Yield successive chunks from the dataset."""
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

In [ ]:
full_payload = [
    {"state": hcp, "questions": questions}
    for hcp in cohort
]

# 2. Process in safe chunks to avoid VRAM crashes
all_predictions = []
CHUNK_SIZE = 8 # Adjust to 64 if you have a high-end GPU (24GB+ VRAM)

for chunk in chunk_data(full_payload, chunk_size=CHUNK_SIZE):
    # Run the batch for the current slice
    batch_results = router.predict_batch(chunk)

    # Save the output dictionaries to our master list
    all_predictions.extend(batch_results)

print(f"Successfully processed {len(all_predictions)} HCP records!")

KeyboardInterrupt: 